# Genesis 19L57G1: Regimes, Observation, Separation, and Invariant Encodings

**AuthorityEffect:** None.  
This notebook provides a complete, mathematically exact walkthrough of the Genesis admission calculus, observation profiles, minimal candidate searches, portable key wire representations, and quotient factorizations from the external `19L57G1` study (`dbccpage/19L57G1@sealed-derivation-v1`).

### Scope and Claim Discipline
- Genesis examines what a formal system can derive when handed an uninterpreted primordial inscription `<>`.
- The inscription `<>` is one opaque datum in the declared source regime. Its visible glyphs do not supply a pair, an ordering, or a relation.
- Any claim that structure is "forced" must state the regime, the permitted changes, the cost measure, the isomorphism convention, and the search bounds.
- All theorems and computations presented here are scoped strictly to the declared finite model.

### Contents
1. **Genesis Inscription & Presentation Grammar (`genesis.ebnf`)**: Token recognition, primitive admission events, and candidate items.
2. **Metatheoretic Regime State Space ($R = (D, F, I, U, Q)$)**: Formal components and fail-closed event refusal via `AdmissionError`.
3. **Inertia & The Supply No-Op Theorem**: Presentation vs. admission and why `<>` alone derives nothing.
4. **Observational Contexts, Plugging, and Total Collapse**: Context counting formula $(n+1)^a - n^a$ and the `collapsed_of_no_insts` theorem.
5. **The Two Readings (Reading L vs Reading F) & The Separation Demand**: Explicit retention ($R_{2L}$) vs. genesis admission ($R_{2F}$).
6. **The Price of Separation is Two**: Cost measures $\mu_1$ vs $\mu_2$ and the `cost1_insufficient_general` proof.
7. **Exhaustive Candidate Search & Isomorphism Classes**: Replaying the 6 vs 10 deltas and proving why $\mu_2$ uniqueness is a declared choice.
8. **Epistemic Firewalls, Canonical Renaming & Frozen Collisions**: Swap symmetries, canonical keys, and deduplication name collisions.
9. **Canonical Portable Key Encoding (`portable_key_v1`)**: The 53-byte wire standard, big-endian words, and boundary probe rejection.
10. **Finite Verification & Quotient Factorization (`genesis_checks`)**: Fiber factorization, monotone pair resolution, and historical ledger audit.
11. **Native Passage State, 2-Cells & Scoped Authorization**: Potentials, commuting squares, and the CL-13 license invariant.
12. **Defensible Claim Register**: What this setting supports, what it excludes, and next tests.

In [1]:
from pathlib import Path
import sys
import itertools

# Dynamically locate package root and bind code paths
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "code/derivation/model.py").is_file())
sys.path.insert(0, str(root / "code"))

from derivation.model import (
    EMPTY, Regime, step, AdmissionError, Supply, AdmitDatum, AdmitForm, AdmitInst, AdmitRuleAll, Impose
)
from derivation.candidates import (
    Bounds, min_responses, NewDatum, NewForm, NewInst, NewRuleAll, apply_delta, response_valid_naive
)
from derivation.observe import contexts, plug, derivable, profiles, satisfies
from derivation.iso import canonical, rename
from portable_key import (
    portable_key_v1, encode_structural_key_v1, decode_portable_key_v1, PREFIX, U64_MAX, PortableKeyError
)
import genesis_checks as checks

out = [
    "Genesis 19L57G1 environment initialized successfully.",
    f"Empty primordial regime: {EMPTY}"
]
print("\n".join(out))


Genesis 19L57G1 environment initialized successfully.
Empty primordial regime: Regime(data=(), forms=(), insts=frozenset(), rules=frozenset(), reqs=frozenset())


## 1. Genesis Inscription & Presentation Grammar (`genesis.ebnf`)

The Genesis presentation language defines three syntactic levels:
1. **Presentation Tokens**:
   - `ground_token`: `<>` (primordial opaque seed)
   - `supplied_token`: `<glyph>` (e.g. `<a>`)
   - `pair_token`: `<glyph glyph>` (e.g. `<ab>`)
   - `cut_token`: `<glyph -> glyph>` (e.g. `<a->b>`)
2. **Admission & Metatheoretic Events**:
   - `supply(token)`: external presentation
   - `admit_datum(token)`: datum internalization
   - `admit_form(arity)`: judgment form declaration ($a \ge 1$)
   - `admit_inst(form, [args...])`: instance assertion
   - `admit_rule_all(form)`: universal all-tuples rule
   - `impose(i, j)`: persistent separation demand
3. **Candidate Response Items ($\Delta$)**:
   - `NewDatum`, `NewForm(arity)`, `NewInst(form, [args...])`, `NewRuleAll(form)`
   - Wrapped in candidate vectors: `delta<item; item; ...>`

The following code implements an exact recursive-descent recognizer for this grammar.

In [2]:
import re

class GenesisGrammar:
    TOKEN_RE = re.compile(r"^<([a-z0-9]|(?:[a-z0-9]{2})|(?:[a-z0-9]->[a-z0-9])|)>$")
    EVENT_RE = re.compile(r"^(supply|admit_datum|admit_form|admit_inst|admit_rule_all|impose)\((.*)\)$")
    
    @classmethod
    def parse_token(cls, text: str) -> str:
        text = text.strip()
        if not cls.TOKEN_RE.match(text):
            raise ValueError(f"Invalid presentation token: {text}")
        if text == "<>":
            return "ground_token"
        inner = text[1:-1]
        if len(inner) == 1:
            return "supplied_token"
        if "->" in inner:
            return "cut_token"
        return "pair_token"
        
    @classmethod
    def parse_event(cls, text: str) -> tuple[str, list[str]]:
        text = text.strip()
        m = cls.EVENT_RE.match(text)
        if not m:
            raise ValueError(f"Invalid genesis event: {text}")
        op, raw_args = m.group(1), m.group(2).strip()
        args = [a.strip() for a in raw_args.split(",")] if raw_args else []
        return op, args

sample_tokens = ["<>", "<a>", "<b>", "<ab>", "<a->b>"]
token_types = {t: GenesisGrammar.parse_token(t) for t in sample_tokens}

out = ["Parsed Presentation Tokens:"]
for t, typ in token_types.items():
    out.append(f"  {t:8s} -> {typ}")

sample_events = [
    "supply(<>)",
    "admit_datum(<a>)",
    "admit_form(1)",
    "admit_inst(0, [0])",
    "admit_rule_all(0)",
    "impose(0, 1)"
]
out.append("")
out.append("Parsed Genesis Events:")
for ev in sample_events:
    op, args = GenesisGrammar.parse_event(ev)
    out.append(f"  {ev:20s} -> op={op}, args={args}")

print("\n".join(out))


Parsed Presentation Tokens:
  <>       -> ground_token
  <a>      -> supplied_token
  <b>      -> supplied_token
  <ab>     -> pair_token
  <a->b>   -> cut_token

Parsed Genesis Events:
  supply(<>)           -> op=supply, args=['<>']
  admit_datum(<a>)     -> op=admit_datum, args=['<a>']
  admit_form(1)        -> op=admit_form, args=['1']
  admit_inst(0, [0])   -> op=admit_inst, args=['0', '[0]']
  admit_rule_all(0)    -> op=admit_rule_all, args=['0']
  impose(0, 1)         -> op=impose, args=['0', '1']


## 2. Metatheoretic Regime State Space ($R = (D, F, I, U, Q)$)

In the formal derivation (`derivation/model.py`), an erased regime is a 5-tuple:
$$R = (D, F, I, U, Q)$$
- $D$: Admitted datum names (tuple of strings, $|D| = n$).
- $F$: Judgment form arities (tuple of positive integers $a_f \ge 1$).
- $I$: Admitted instances (frozenset of $(f, \vec{a})$ where $0 \le f < |F|$ and $\vec{a} \in D^{a_f}$).
- $U$: Uniform all-tuples rules (frozenset of forms $f$ where all combinations are derivable).
- $Q$: Persistent separation requirements (frozenset of unordered pairs $(i, j)$ with $0 \le i < j < n$).

### Fail-Closed Event Refusal
Metatheoretic transitions are governed by `step(r, event) -> Regime`.
If an event references an unadmitted datum, an undeclared form, an arity mismatch, or an invalid form arity ($a=0$), the transition **fails closed** and raises `AdmissionError`.
This refusal prevents the silent smuggling of missing ontology.

In [3]:
# 1. Genesis starts empty
r0 = EMPTY
out = [
    f"Initial state r0: data={r0.data}, forms={r0.forms}, insts={r0.insts}, rules={r0.rules}, reqs={r0.reqs}"
]

# 2. Refusal tests on empty regime: ill-formed admissions raise AdmissionError
for bad_event, desc in [
    (AdmitForm(0), "Arity 0"),
    (AdmitInst(0, (0,)), "Undeclared form 0"),
    (Impose(0, 1), "Imposing on absent data 0, 1"),
]:
    try:
        step(r0, bad_event)
        assert False, f"{desc} must raise AdmissionError"
    except AdmissionError as exc:
        out.append(f"  Refused {desc}: {exc}")
out.append("Fail-closed refusal verified: ill-formed admissions raise AdmissionError.")

# 3. Methodical valid construction
r1 = step(r0, AdmitDatum("a"))
assert r1.n_data() == 1

r2 = step(r1, AdmitForm(1))  # Unary form 0
assert len(r2.forms) == 1 and r2.forms == (1,)

r3 = step(r2, AdmitInst(0, (0,)))  # Assert Form 0 on Datum 0
assert (0, (0,)) in r3.insts

r4 = step(r3, AdmitDatum("b"))  # Add second datum 'b'
assert r4.n_data() == 2

r5 = step(r4, Impose(0, 1))  # Require that Datum 0 and Datum 1 be separated
assert (0, 1) in r5.reqs

out.append(f"Constructed regime r5: n_data={r5.n_data()}, forms={r5.forms}, insts={r5.insts}, reqs={r5.reqs}")
print("\n".join(out))


Initial state r0: data=(), forms=(), insts=frozenset(), rules=frozenset(), reqs=frozenset()
  Refused Arity 0: form arity must be >= 1, got 0
  Refused Undeclared form 0: unknown form 0
  Refused Imposing on absent data 0, 1: requirement out of range: (0, 1)
Fail-closed refusal verified: ill-formed admissions raise AdmissionError.
Constructed regime r5: n_data=2, forms=(1,), insts=frozenset({(0, (0,))}), reqs=frozenset({(0, 1)})


## 3. Inertia & The Supply No-Op Theorem

A primary principle of the Genesis study is that **presentation is not admission**.

In Lean 4, this is the theorem `supply_no_op`:
$$\text{stepE}(r, \text{supply}(t)) = \text{some}(r)$$

When an agent or environment supplies a token, the internal regime remains unchanged.
The primordial seed `<>` does not generate points, metrics, or categories on its own:
> *"Result: **inertia**. Exactly nothing follows from the initial inscription alone."*  
> — 19L57G1 `paper.md`, §2

In [4]:
# Test supply on empty regime
s_empty = step(EMPTY, Supply("<>"))
assert s_empty == EMPTY, "Supply('<>') must leave EMPTY unchanged"

# Test supply on constructed regime r5
s_r5 = step(r5, Supply("<a>"))
assert s_r5 == r5, "Supply('<a>') must leave r5 unchanged"

# Supplying any token string is an external presentation no-op
for token in ["<>", "<a>", "<b>", "<ab>", "<a->b>"]:
    res = step(EMPTY, Supply(token))
    assert res == EMPTY

out = [
    "Inertia check 1: step(EMPTY, Supply('<>')) == EMPTY [PASS]",
    "Inertia check 2: step(r5, Supply('<a>')) == r5 [PASS]",
    "Inertia check 3: All presentation tokens preserve the empty state without side effects."
]
print("\n".join(out))


Inertia check 1: step(EMPTY, Supply('<>')) == EMPTY [PASS]
Inertia check 2: step(r5, Supply('<a>')) == r5 [PASS]
Inertia check 3: All presentation tokens preserve the empty state without side effects.


## 4. Observational Contexts, Plugging, and Total Collapse

How does a regime observe its data?

1. **Contexts**: A context $c = (f, \vec{p})$ consists of a form index $f$ and a tuple of length $\text{arity}(f)$, where at least one position is a hole (`None`) and remaining positions are bound to datum indices $0 \le d < n$.
2. **Context Counting Formula**:
   For $n$ data and a single form of arity $a$:
   $$|C(n, a)| = (n + 1)^a - n^a$$
   *Proof*: Each argument position has $n+1$ options ($n$ data plus the hole `None`). Total patterns = $(n+1)^a$. Exactly $n^a$ patterns contain no holes. Since a context must contain at least one hole, subtracting gives $(n+1)^a - n^a$.
3. **Plugging**: `plug(c, d)` replaces every hole in $c$ with the same datum index $d$.
4. **Derivability**: $R \vdash (f, \vec{a}) \iff (f, \vec{a}) \in I \lor f \in U$.
5. **Observational Profile**: $\text{profile}(R, d) = \{c \in C(R) \mid R \vdash \text{plug}(c, d)\}$.
6. **Total Collapse (`collapsed_of_no_insts`)**:
   If a regime has no instances and no uniform rules ($I = \emptyset, U = \emptyset$), then for every context $c$ and datum $d$, $\text{plug}(c, d) \notin I$.
   Thus $\text{profile}(R, d) = \emptyset$ for every datum!
   Admitting new data without instances leaves them all observationally indistinguishable.

In [5]:
# 1. Verify Context Counting Formula across 35 cases (n in 0..4, arity in 1..7)
count_checks = 0
for n in range(5):
    for arity in range(1, 8):
        r = Regime(tuple(f"d{i}" for i in range(n)), (arity,), frozenset(), frozenset(), frozenset())
        actual_contexts = contexts(r)
        expected_count = (n + 1) ** arity - n ** arity
        assert len(actual_contexts) == expected_count, f"Mismatch at n={n}, arity={arity}"
        count_checks += 1

# 2. Demonstrate Total Collapse (collapsed_of_no_insts)
# Regime with 3 data, 1 binary form, but zero instances
r_collapsed = Regime(("x", "y", "z"), (2,), frozenset(), frozenset(), frozenset())
cs = contexts(r_collapsed)
ps = profiles(r_collapsed)
assert ps[0] == ps[1] == ps[2] == frozenset()

out = [
    f"Verified context counting formula across {count_checks} distinct (n, arity) configurations.",
    "",
    f"Regime with 3 data and binary form: {len(cs)} well-formed contexts.",
    f"Profile for x (0): {ps[0]}",
    f"Profile for y (1): {ps[1]}",
    f"Profile for z (2): {ps[2]}",
    "Theorem confirmed: Without instances, all admitted data have empty profiles (collapsed)."
]
print("\n".join(out))


Verified context counting formula across 35 distinct (n, arity) configurations.

Regime with 3 data and binary form: 7 well-formed contexts.
Profile for x (0): frozenset()
Profile for y (1): frozenset()
Profile for z (2): frozenset()
Theorem confirmed: Without instances, all admitted data have empty profiles (collapsed).


## 5. The Two Readings (Reading L vs Reading F) & The Separation Demand

The Genesis study identifies an ambiguity in whether the initial inscription `<>` is retained as an admitted datum:

- **Reading L (Explicit Retention)**:
  Only explicit admissions create data. Genesis begins empty, then admits `a` and `b`.
  Datum count: 2.
  Target pair: $(0, 1)$.
  Base regime: $R_{2L} = \text{Regime}(("a", "b"), (), \emptyset, \emptyset, \{(0, 1)\})$.
- **Reading F (Inscription Retention)**:
  The initial inscription `<>` is retained as datum 0, followed by `a` and `b`.
  Datum count: 3.
  Target pair: $(1, 2)$.
  Base regime: $R_{2F} = \text{Regime}(("<>", "a", "b"), (), \emptyset, \emptyset, \{(1, 2)\})$.

The literature does not adjudicate between these readings; both are carried throughout the derivation.

In [6]:
# Define base regimes for Reading L and Reading F
R2L = Regime(("a", "b"), (), frozenset(), frozenset(), frozenset({(0, 1)}))
R2F = Regime(("<>", "a", "b"), (), frozenset(), frozenset(), frozenset({(1, 2)}))

out = [
    "Base Regime Reading L:",
    f"  Data: {R2L.data} (count={R2L.n_data()})",
    f"  Protected Separation Target: {list(R2L.reqs)[0]}",
    f"  Satisfied initially: {satisfies(R2L)} (Collapsed)",
    "",
    "Base Regime Reading F:",
    f"  Data: {R2F.data} (count={R2F.n_data()})",
    f"  Protected Separation Target: {list(R2F.reqs)[0]}",
    f"  Satisfied initially: {satisfies(R2F)} (Collapsed)"
]
print("\n".join(out))


Base Regime Reading L:
  Data: ('a', 'b') (count=2)
  Protected Separation Target: (0, 1)
  Satisfied initially: False (Collapsed)

Base Regime Reading F:
  Data: ('<>', 'a', 'b') (count=3)
  Protected Separation Target: (1, 2)
  Satisfied initially: False (Collapsed)


## 6. The Price of Separation is Two

When a regime must separate a target pair $(i, j)$, an external delta candidate $\Delta$ is proposed.

### Cost Measures
- $\mu_1(\Delta) = |\Delta|$ (item count)
- $\mu_2(\Delta) = (|\Delta|, \sum_{f \in \Delta} \text{arity}(f))$ (lexicographic comparison: fewest items, then minimal arity sum).

### The Cost-1 Insufficiency Theorem (`cost1_insufficient_general`)
**Theorem**: If $R$ has 0 forms and 0 instances, no candidate $\Delta$ with $|\Delta| \le 1$ can separate $(i, j)$.
*Proof*:
1. $|\Delta| = 0$: $R$ is unchanged $\implies$ still collapsed.
2. $|\Delta| = 1$:
   - $\Delta = [\text{NewDatum}]$: adds an inert datum; observations remain empty $\implies$ collapsed.
   - $\Delta = [\text{NewForm}(a)]$: declares a form arity $a$, but adds 0 instances $\implies$ all derivabilities remain False $\implies$ collapsed.
   - $\Delta = [\text{NewInst}(f, \dots)]$: invalid; no form $f$ exists in $R \implies$ refused (`AdmissionError`).
   - $\Delta = [\text{NewRuleAll}(f)]$: invalid; no form $f$ exists $\implies$ refused (`AdmissionError`).

Thus, separation requires at least 2 items: declare a form, then assert an instance.
The value 2 is an exact property of the **declare-then-assert grammar**.

In [7]:
# 1. Demonstrate Cost-1 Insufficiency on Reading L
one_item_candidates = [
    (NewDatum(),),
    (NewForm(1),),
    (NewForm(2),),
]

out = ["Testing 1-item candidates on Reading L:"]
for delta in one_item_candidates:
    valid = response_valid_naive(R2L, (0, 1), delta)
    out.append(f"  Delta {delta}: valid={valid}")
    assert not valid, "No 1-item candidate may separate R2L"

# 2. A 2-item candidate that succeeds: declare unary form, assert on datum 0
delta_u0 = (NewForm(1), NewInst(0, (0,)))
assert response_valid_naive(R2L, (0, 1), delta_u0)

r_sep = apply_delta(R2L, delta_u0)
assert satisfies(r_sep)
ps_sep = profiles(r_sep)

out.append("")
out.append(f"Cost-2 Candidate {delta_u0} successfully separates Reading L:")
out.append(f"  Profile Datum 0 ('a'): {ps_sep[0]}")
out.append(f"  Profile Datum 1 ('b'): {ps_sep[1]}")
out.append(f"  Requirement (0, 1) satisfied: {satisfies(r_sep)}")

print("\n".join(out))


Testing 1-item candidates on Reading L:
  Delta (NewDatum(),): valid=False
  Delta (NewForm(arity=1),): valid=False
  Delta (NewForm(arity=2),): valid=False

Cost-2 Candidate (NewForm(arity=1), NewInst(form=0, args=(0,))) successfully separates Reading L:
  Profile Datum 0 ('a'): frozenset({(0, (None,))})
  Profile Datum 1 ('b'): frozenset()
  Requirement (0, 1) satisfied: True


## 7. Exhaustive Candidate Search & Isomorphism Classes

Let us execute the exhaustive search `min_responses` on Reading L and Reading F within bounds:
`max_items = 2`, `max_arity = 2`.

### Theoretical Findings
- **Under $\mu_1$**:
  - Reading L: 6 valid deltas in 3 isomorphism classes:
    1. Unary mark: $(1,)$ on datum 0 or datum 1 (2 deltas)
    2. Binary directed link: $(2,)$ on $(0, 1)$ or $(1, 0)$ (2 deltas)
    3. Binary self-loop: $(2,)$ on $(0, 0)$ or $(1, 1)$ (2 deltas)
  - Reading F: 10 valid deltas in 5 isomorphism classes.
  - Notice: for any arity $k$, asserting a $k$-ary relation with $(0, \dots, 0)$ costs 2 items!
    Thus, under $\mu_1$, the minimal tie class is **countably infinite**.
- **Under $\mu_2$**:
  - The arity sum is added to the metric:
    - Unary mark: cost = $(2, 1)$
    - Binary link / self-loop: cost = $(2, 2)$
  - Lexicographic ordering $(2, 1) < (2, 2)$ uniquely selects the unary mark!
  - Both Reading L and Reading F collapse to **exactly 1 isomorphism class** (the unary mark).
- **Epistemic Result**: Choosing $\mu_2$ is a declared premise. Uniqueness is a result of choosing $\mu_2$, not an endogenous theorem of nature.

In [8]:
# Run exhaustive candidate search across both readings and measures
bounds = Bounds(max_items=2, max_arity=2)
search_results = []

for reading, base, target in [("L", R2L, (0, 1)), ("F", R2F, (1, 2))]:
    for measure in ["mu1", "mu2"]:
        res = min_responses(base, target, measure, bounds)
        assert res is not None
        # Check canonical isomorphism classes
        classes = {canonical(apply_delta(base, d)) for d in res.deltas}
        row = [reading, measure, res.cost[0], res.cost[1], len(res.deltas), len(classes)]
        search_results.append(row)

out = [
    "Exhaustive Minimum Search Census:",
    f"{'Reading':<8} {'Measure':<8} {'Cost':<12} {'Valid Deltas':<14} {'Iso Classes':<12}",
    "-" * 56
]
for row in search_results:
    cost_str = f"({row[2]}, {row[3]})" if row[3] != -1 else str(row[2])
    out.append(f"{row[0]:<8} {row[1]:<8} {cost_str:<12} {row[4]:<14} {row[5]:<12}")

expected_census = [
    ["L", "mu1", 2, -1, 6, 3],
    ["L", "mu2", 2, 1, 2, 1],
    ["F", "mu1", 2, -1, 10, 5],
    ["F", "mu2", 2, 1, 2, 1]
]
assert search_results == expected_census, "Census must match 19L57G1 exactly"
out.append("")
out.append("All class counts match the proved 19L57G1 census.")

print("\n".join(out))


Exhaustive Minimum Search Census:
Reading  Measure  Cost         Valid Deltas   Iso Classes 
--------------------------------------------------------
L        mu1      2            6              3           
L        mu2      (2, 1)       2              1           
F        mu1      2            10             5           
F        mu2      (2, 1)       2              1           

All class counts match the proved 19L57G1 census.


## 8. Epistemic Firewalls, Canonical Renaming & Frozen Collisions

### Epistemic Firewall Normal Form (EFNF)
The canonicalizer `canonical(r)` computes the structural canonical form modulo datum permutations and form re-indexing.
Two regimes are isomorphic if and only if their canonical forms match.

### Swap Symmetry
For Reading L with target $(0, 1)$, the two $\mu_2$ deltas are:
- $\Delta_0 = [\text{NewForm}(1), \text{NewInst}(0, (0,))]$ (mark datum 0)
- $\Delta_1 = [\text{NewForm}(1), \text{NewInst}(0, (1,))]$ (mark datum 1)

These two states are isomorphic under the transposition $\tau = (0\;1)$.
No swap-respecting rule can prefer $\Delta_0$ over $\Delta_1$ without an arbitrary external bias.

### Frozen Naming Collision
If an implementation uses a fixed string generator (e.g. `"w1"`), a collision occurs if that name was already internalized:
`apply_delta(step(EMPTY, AdmitDatum("w1")), (NewDatum(),))` yields no change under set deduplication!
The companion engine must enforce freshness and preserve refusal outcomes.

In [9]:
# 1. Verify swap symmetry: Delta_0 and Delta_1 yield identical canonical forms
r_delta0 = apply_delta(R2L, (NewForm(1), NewInst(0, (0,))))
r_delta1 = apply_delta(R2L, (NewForm(1), NewInst(0, (1,))))

canon0 = canonical(r_delta0)
canon1 = canonical(r_delta1)
assert canon0 == canon1, "Delta 0 and Delta 1 must have identical canonical representations"

# 2. Frozen naming collision reproduction
r_w1 = step(EMPTY, AdmitDatum("w1"))
r_w1_bump = apply_delta(r_w1, (NewDatum(),))
assert r_w1_bump == r_w1, "Frozen datum naming collision must reproduce"

# 3. Uniform rule collapse
assert satisfies(r_delta0)
r_collapsed_rule = step(r_delta0, AdmitRuleAll(0))
assert not satisfies(r_collapsed_rule), "Uniform rule must destroy observational separation"
rule_profs = profiles(r_collapsed_rule)

out = [
    "Swap symmetry confirmed: canon(R + Delta_0) == canon(R + Delta_1).",
    "Frozen naming collision reproduced: deduplication without fresh names drops the addition.",
    f"Uniform rule collapse: profiles equal = {rule_profs[0] == rule_profs[1]}"
]
print("\n".join(out))


Swap symmetry confirmed: canon(R + Delta_0) == canon(R + Delta_1).
Frozen naming collision reproduced: deduplication without fresh names drops the addition.
Uniform rule collapse: profiles equal = True


## 9. Canonical Portable Key Encoding (`portable_key_v1`)

The portable key is the binary wire format for exchanging regime states across language boundaries (Python, Rust, Lean 4):
- Header: 13 bytes `b"GENESIS-KEY\x00\x01"`.
- Structural Words: 40 bytes (5 big-endian 64-bit integers):
  1. $n_{\text{data}}$
  2. $|F|$
  3. $|I|$
  4. $|U|$
  5. $|Q|$
- Data Payload: Followed by big-endian words for form arities, instances, rules, and requirements.
- Total length of `EMPTY` regime key: exactly 53 bytes ($13 + 40$).

### Fail-Closed Boundary Vectors
The decoder `decode_portable_key_v1` strictly rejects:
- Truncated bytes
- Corrupted magic headers
- Out-of-bounds indices ($u64$ overflow, negative words, indices $\ge n$)
- Reparse points / symlinks

In [10]:
# 1. Verify Empty Regime 53-byte wire vector
empty_key = portable_key_v1(EMPTY)
assert len(empty_key) == 53
assert empty_key.startswith(b"GENESIS-KEY\x00\x01")
assert empty_key[13:] == bytes(40)  # 5 zero u64 words

# 2. Round-trip structural encoding/decoding
key_struct = (2, (1, 2), ((0, (1,)), (1, (0, 1))), (0, 1), ((0, 1),))
wire_bytes = encode_structural_key_v1(key_struct)
decoded = decode_portable_key_v1(wire_bytes)
assert decoded == key_struct, "Roundtrip wire decode mismatch"

# 3. Fail-closed rejection of corrupted probes
corrupted_magic = b"WRONG-HEADER" + empty_key[12:]
try:
    decode_portable_key_v1(corrupted_magic)
    assert False, "Must fail"
except PortableKeyError:
    pass

truncated_key = empty_key[:30]
try:
    decode_portable_key_v1(truncated_key)
    assert False, "Must fail"
except PortableKeyError:
    pass

out = [
    f"Empty regime key: {empty_key.hex()} (len={len(empty_key)} bytes)",
    "Structural key roundtrip encoding/decoding verified.",
    "Corrupted header rejected fail-closed [PASS]",
    "Truncated key rejected fail-closed [PASS]"
]
print("\n".join(out))


Empty regime key: 47454e455349532d4b4559000100000000000000000000000000000000000000000000000000000000000000000000000000000000 (len=53 bytes)
Structural key roundtrip encoding/decoding verified.
Corrupted header rejected fail-closed [PASS]
Truncated key rejected fail-closed [PASS]


## 10. Finite Verification & Quotient Factorization (`genesis_checks`)

In `genesis_checks.py`, the research package ships finite verification routines that test quotient factorizations, history adequacy, and pair resolution:

1. **Quotient Factorization**:
   $$f: X \to Y \quad \text{factors through} \quad q: X \to Q$$
   If two points $x_1, x_2$ map to the same quotient fiber $q(x_1) = q(x_2)$ but have conflicting values $f(x_1) \ne f(x_2)$, no well-defined descent exists. The auditor refuses by returning `None`.
2. **Monotone Pair Resolution**:
   For $n$ data, there are $M(n) = \frac{n(n - 1)}{2}$ candidate pairs.
   Resolving differences reduces unresolved pairs monotonically down to 0.
3. **Claim Ledger Audit**:
   Verification of the 12 historical claim records without promotional hype.

In [11]:
# Execute shipped Paper 000 finite verifier
repro_report = checks.run_reproduction()
assert repro_report["ok"] is True, "All shipped finite checks must pass"

out = ["Genesis Finite Verification Report:"]
for k, v in repro_report.items():
    if k != "checked_ids":
        out.append(f"  {k:24s}: {v}")
out.append("")
out.append("All 10 finite verification invariants certified PASS.")

print("\n".join(out))


Genesis Finite Verification Report:
  ok                      : True
  collapsed_quotient      : None
  separated_quotient      : {'qx': 'yes', 'qy': 'no'}
  insufficient_history    : None
  sufficient_history      : {'v1': 's1', 'v2': 's2'}
  single_admission        : {'datum': False, 'form': False, 'inst': False}
  form_then_instance      : True
  pair_rounds             : {'0': 0, '1': 0, '2': 1, '3': 3, '4': 6}
  endpoints_distinct      : True

All 10 finite verification invariants certified PASS.


## 11. Native Passage State, 2-Cells & Scoped Authorization

In the larger Omega Engine architecture (Levels 29, 65, 80), the admission regime is contrasted with the **Passage state**:
- Passage states maintain operational tracking: occurrences, assertions, witnesses, protections, observations, budgets, and licenses.
- For an integer potential $\phi$, the edge cost along transition $e$ is:
  $$d\phi(s, e) = \phi(T_e(s)) - \phi(s)$$
- For a two-step square (2-cell), the defect between paths is:
  $$d_\square(d\phi)(s; e, f) = \phi(T_f(T_e(s))) - \phi(T_e(T_f(s)))$$
- When steps commute ($T_f \circ T_e = T_e \circ T_f$), the 2-cell boundary vanishes:
  $$d_\square(d\phi) = 0$$
- **Scoped Authorization Invariant (CL-13)**:
  Any quantity preserved by every valid event retains its genesis value across all reachable states.
  Licenses must be bound to parent histories, targets, and bounds; license reuse is forbidden.

In [12]:
# Executable simulation of 2-cell commutation and potential defect
def phi_data_count(r: Regime) -> int:
    return r.n_data()

def phi_form_count(r: Regime) -> int:
    return len(r.forms)

# Commuting pair: AdmitDatum('x') and AdmitForm(1)
s = EMPTY
e1 = AdmitDatum("x")
e2 = AdmitForm(1)

# Path 1: e1 then e2
s_1 = step(s, e1)
s_12 = step(s_1, e2)

# Path 2: e2 then e1
s_2 = step(s, e2)
s_21 = step(s_2, e1)

assert s_12 == s_21, "AdmitDatum and AdmitForm commute"

# Evaluate 2-cell defect for phi_data_count
d_square = phi_data_count(s_12) - phi_data_count(s_21)
assert d_square == 0

out = [
    "Commuting 2-cell defect d_square(d phi) = 0 verified.",
    f"Endpoint state reached via both paths: n_data={s_12.n_data()}, forms={s_12.forms}"
]
print("\n".join(out))


Commuting 2-cell defect d_square(d phi) = 0 verified.
Endpoint state reached via both paths: n_data=1, forms=(1,)


## 12. Defensible Claim Register

In accordance with repository claim hygiene, we record the exact scope of established facts:

### 1. What This Notebook Supports
- In the declared declare-then-assert formal model, the primordial inscription `<>` is opaque; `step(r, Supply("<>")) == r` produces no structural change (inertia).
- Observational collapse holds for any regime with no instances (`collapsed_of_no_insts`).
- The context counting formula $|C(n, a)| = (n+1)^a - n^a$ holds across all tested $(n, a)$.
- The price of separation is at least 2 (`cost1_insufficient_general`).
- Under metric $\mu_1$, Reading L has 6 valid deltas in 3 classes; Reading F has 10 valid deltas in 5 classes.
- Under metric $\mu_2$, both readings have 2 valid deltas in exactly 1 isomorphism class (the unary mark).
- Portable binary keys serialize canonical regimes deterministically into a 53-byte base representation.

### 2. What This Notebook Does NOT Support
- That `<>` intrinsically "contains" geometry, pairs, physics, or quantum states.
- That the choice of metric $\mu_2$ is an unforced law of the universe (it is an explicit external preference for lower arity).
- That toy mathematical derivations prove empirical capabilities or limits of large language models.
- That the 19 primitive families are complete or prove quantum gravity / Regge calculus theorems.

### 3. Narrowest Defensible Claim
Under the declare-then-assert grammar and regime definition $R = (D, F, I, U, Q)$, one opaque token `<>` derives nothing, separating a pair requires at least two operations, and lexicographic arity minimization ($\mu_2$) selects a unique unary mark up to swap isomorphism.

### 4. Next Test Needed
Automated dual-harness verification comparing Python `portable_key_v1` byte sequences with Lean 4 `wireReplay` binaries in a hermetic CI container.